# 3. Baseline Transformer: Genre-Conditioned Story Generation

This notebook builds and trains a **vanilla encoder-decoder Transformer** for genre-conditioned story generation, consuming the artifacts produced by notebook 2 (Preprocessing).

**Pipeline:**
1. Load preprocessed artifacts (tokenizer, splits, config)
2. Build PyTorch `Dataset` & `DataLoader` with truncation
3. Define encoder-decoder Transformer architecture
4. Train with teacher forcing + cross-entropy loss
5. Track all metrics via **Weights & Biases**
6. Evaluate on validation set (loss & perplexity)
7. Generate sample stories (greedy, top-k, nucleus)
8. Save best checkpoint as W&B artifact

**Proposal references:** Sections 4 (Model), 5 (Training), 6.1 (W&B tracking), 6.3 (Model versioning), 9 (Baseline experiment)

## 1. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer
from pathlib import Path
import json
import pickle
import math
import time
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (14, 5)

try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    WANDB_AVAILABLE = False
    print("wandb not installed \u2014 metrics will only be logged locally.")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print("All imports loaded.")

## 2. Baseline Configuration

All hyperparameters live in a single dict so they are logged to W&B verbatim and can be reproduced.

In [ ]:
CONFIG = {
    # Model architecture
    "d_model": 256,
    "nhead": 8,
    "num_encoder_layers": 3,
    "num_decoder_layers": 3,
    "dim_feedforward": 512,
    "dropout": 0.1,

    # Sequence limits
    "max_enc_len": 64,
    "max_dec_len": 512,

    # Training
    "batch_size": 8,
    "learning_rate": 3e-4,
    "epochs": 15,
    "label_smoothing": 0.1,
    "grad_clip": 1.0,
    "warmup_steps": 200,

    # Misc
    "seed": SEED,
    "experiment_name": "baseline-v1",
}

print("Baseline configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## 3. Load Preprocessed Artifacts

Everything needed was saved by notebook 2 into `data/processed/`.

In [ ]:
DATA_DIR = Path("../data/processed")

with open(DATA_DIR / "preprocessing_config.json") as f:
    prep_config = json.load(f)

tokenizer = GPT2Tokenizer.from_pretrained(str(DATA_DIR / "tokenizer"))

with open(DATA_DIR / "genre2id.json") as f:
    genre2id = json.load(f)
with open(DATA_DIR / "id2genre.json") as f:
    id2genre = {int(k): v for k, v in json.load(f).items()}

VOCAB_SIZE = prep_config["vocab_size"]
PAD_ID = prep_config["special_tokens"]["pad_id"]
BOS_ID = prep_config["special_tokens"]["bos_id"]
EOS_ID = prep_config["special_tokens"]["eos_id"]
NUM_GENRES = prep_config["num_genres"]

CONFIG["vocab_size"] = VOCAB_SIZE
CONFIG["pad_id"] = PAD_ID
CONFIG["num_genres"] = NUM_GENRES

def load_split(name):
    with open(DATA_DIR / f"{name}.pkl", "rb") as f:
        return pickle.load(f)

train_data = load_split("train")
val_data = load_split("val")
test_data = load_split("test")

print(f"Vocab size : {VOCAB_SIZE}")
print(f"Genres     : {NUM_GENRES}")
print(f"Special IDs: BOS={BOS_ID}  EOS={EOS_ID}  PAD={PAD_ID}")
print(f"Splits     : Train={len(train_data['encoder_ids'])}  "
      f"Val={len(val_data['encoder_ids'])}  "
      f"Test={len(test_data['encoder_ids'])}")

## 4. Dataset & DataLoader

Sequences are **truncated** to `max_enc_len` / `max_dec_len` at collate time and **dynamically padded** per batch.

In [ ]:
class StoryDataset(Dataset):
    """Wraps the pickle-saved split dicts."""

    def __init__(self, data_dict):
        self.encoder_ids = data_dict["encoder_ids"]
        self.decoder_ids = data_dict["decoder_ids"]
        self.genre_ids = data_dict["genre_ids"]

    def __len__(self):
        return len(self.encoder_ids)

    def __getitem__(self, idx):
        return {
            "encoder_ids": self.encoder_ids[idx],
            "decoder_ids": self.decoder_ids[idx],
            "genre_id": self.genre_ids[idx],
        }


def collate_fn(batch):
    """Truncate, pad, and build masks for one batch."""
    max_enc = CONFIG["max_enc_len"]
    max_dec = CONFIG["max_dec_len"]

    enc_seqs = [torch.tensor(b["encoder_ids"][:max_enc], dtype=torch.long) for b in batch]
    dec_seqs = [torch.tensor(b["decoder_ids"][:max_dec], dtype=torch.long) for b in batch]
    genre_ids = torch.tensor([b["genre_id"] for b in batch], dtype=torch.long)

    enc_padded = nn.utils.rnn.pad_sequence(enc_seqs, batch_first=True, padding_value=PAD_ID)
    dec_padded = nn.utils.rnn.pad_sequence(dec_seqs, batch_first=True, padding_value=PAD_ID)

    enc_pad_mask = (enc_padded == PAD_ID)
    dec_pad_mask = (dec_padded == PAD_ID)

    return {
        "encoder_ids": enc_padded,
        "encoder_pad_mask": enc_pad_mask,
        "decoder_ids": dec_padded,
        "decoder_pad_mask": dec_pad_mask,
        "genre_ids": genre_ids,
    }


train_ds = StoryDataset(train_data)
val_ds = StoryDataset(val_data)
test_ds = StoryDataset(test_data)

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False, collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)}  |  Val: {len(val_loader)}  |  Test: {len(test_loader)}")

batch = next(iter(train_loader))
print("\nSample batch shapes:")
for k, v in batch.items():
    print(f"  {k:20s} \u2192 {tuple(v.shape)}")

## 5. Model Architecture

A vanilla **encoder-decoder Transformer** (proposal \u00a74).

- Shared token embedding between encoder and decoder
- Sinusoidal positional encoding
- Genre conditioning via the `[GENRE: <label>]` prefix already baked into encoder inputs

| Component | Detail |
|---|---|
| Embedding | shared, `vocab_size \u00d7 d_model` |
| Positional Encoding | sinusoidal (fixed) |
| Encoder | `num_encoder_layers` standard layers |
| Decoder | `num_decoder_layers` layers with causal + cross-attention |
| Output head | linear projection to `vocab_size` |

In [ ]:
class PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding."""

    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class StoryTransformer(nn.Module):
    """
    Encoder-decoder Transformer for genre-conditioned story generation.

    Encoder receives  [BOS] [GENRE: label] title [EOS]
    Decoder receives  [BOS] story_tokens ...           (teacher-forced)
    Output predicts   story_tokens ... [EOS]           (shifted by 1)
    """

    def __init__(self, vocab_size, d_model, nhead, num_enc_layers,
                 num_dec_layers, dim_ff, dropout, pad_id, max_len=5000):
        super().__init__()
        self.d_model = d_model
        self.pad_id = pad_id

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_encoder = PositionalEncoding(d_model, max_len, dropout)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_enc_layers,
            num_decoder_layers=num_dec_layers,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
        )

        self.output_proj = nn.Linear(d_model, vocab_size)
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def _embed(self, token_ids):
        return self.pos_encoder(self.embedding(token_ids) * math.sqrt(self.d_model))

    def encode(self, src, src_key_padding_mask=None):
        return self.transformer.encoder(
            self._embed(src), src_key_padding_mask=src_key_padding_mask
        )

    def decode(self, tgt, memory, tgt_mask=None,
               tgt_key_padding_mask=None, memory_key_padding_mask=None):
        return self.transformer.decoder(
            self._embed(tgt), memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask,
        )

    def forward(self, src, tgt, src_key_padding_mask=None,
                tgt_key_padding_mask=None, tgt_mask=None):
        memory = self.encode(src, src_key_padding_mask)
        dec_out = self.decode(
            tgt, memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask,
        )
        return self.output_proj(dec_out)


def create_causal_mask(sz, device):
    return nn.Transformer.generate_square_subsequent_mask(sz, device=device)


model = StoryTransformer(
    vocab_size=VOCAB_SIZE,
    d_model=CONFIG["d_model"],
    nhead=CONFIG["nhead"],
    num_enc_layers=CONFIG["num_encoder_layers"],
    num_dec_layers=CONFIG["num_decoder_layers"],
    dim_ff=CONFIG["dim_feedforward"],
    dropout=CONFIG["dropout"],
    pad_id=PAD_ID,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable   = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model on {DEVICE}")
print(f"  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable:,}")

## 6. Optimizer, Scheduler & Loss

Per proposal \u00a75:
- **AdamW** with \u03b2=(0.9, 0.98)
- **Linear warmup** then constant LR
- **Label-smoothed cross-entropy** (padding tokens ignored)
- **Gradient clipping** at 1.0

In [ ]:
optimizer = optim.AdamW(
    model.parameters(), lr=CONFIG["learning_rate"],
    betas=(0.9, 0.98), eps=1e-9,
)

def warmup_then_constant(step):
    if step < CONFIG["warmup_steps"]:
        return step / max(1, CONFIG["warmup_steps"])
    return 1.0

scheduler = optim.lr_scheduler.LambdaLR(optimizer, warmup_then_constant)

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID,
    label_smoothing=CONFIG["label_smoothing"],
)

print(f"Optimizer : AdamW (lr={CONFIG['learning_rate']})")
print(f"Scheduler : linear warmup ({CONFIG['warmup_steps']} steps)")
print(f"Loss      : CrossEntropy (label_smoothing={CONFIG['label_smoothing']}, ignore PAD)")
print(f"Grad clip : {CONFIG['grad_clip']}")

## 7. W&B Initialization

All metrics, hyperparameters, and generated samples are logged to Weights & Biases (proposal \u00a76.1).

In [ ]:
USE_WANDB = WANDB_AVAILABLE

if USE_WANDB:
    wandb.init(
        project="genre-story-generator",
        name=CONFIG["experiment_name"],
        config=CONFIG,
        tags=["baseline", "encoder-decoder", "transformer"],
    )
    print("W&B run initialized.")
else:
    print("W&B disabled \u2014 logging locally only.")

## 8. Training & Validation Functions

**Teacher forcing** (proposal \u00a75): at each decoder step the ground-truth previous token is fed as input.

```
decoder input  = [BOS] tok_1  tok_2  ...  tok_{T-1}
decoder target =  tok_1  tok_2  tok_3  ...  tok_T
```

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0.0
    total_tokens = 0

    for batch in loader:
        enc_ids      = batch["encoder_ids"].to(device)
        dec_ids      = batch["decoder_ids"].to(device)
        enc_pad_mask = batch["encoder_pad_mask"].to(device)
        dec_pad_mask = batch["decoder_pad_mask"].to(device)

        tgt_input  = dec_ids[:, :-1]
        tgt_output = dec_ids[:, 1:]
        tgt_pad    = dec_pad_mask[:, :-1]

        tgt_mask = create_causal_mask(tgt_input.size(1), device)

        logits = model(
            enc_ids, tgt_input,
            src_key_padding_mask=enc_pad_mask,
            tgt_key_padding_mask=tgt_pad,
            tgt_mask=tgt_mask,
        )

        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_output.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
        optimizer.step()
        scheduler.step()

        non_pad = (tgt_output != PAD_ID).sum().item()
        total_loss += loss.item() * non_pad
        total_tokens += non_pad

    avg_loss = total_loss / max(total_tokens, 1)
    ppl = math.exp(min(avg_loss, 100))
    return avg_loss, ppl


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    for batch in loader:
        enc_ids      = batch["encoder_ids"].to(device)
        dec_ids      = batch["decoder_ids"].to(device)
        enc_pad_mask = batch["encoder_pad_mask"].to(device)
        dec_pad_mask = batch["decoder_pad_mask"].to(device)

        tgt_input  = dec_ids[:, :-1]
        tgt_output = dec_ids[:, 1:]
        tgt_pad    = dec_pad_mask[:, :-1]

        tgt_mask = create_causal_mask(tgt_input.size(1), device)

        logits = model(
            enc_ids, tgt_input,
            src_key_padding_mask=enc_pad_mask,
            tgt_key_padding_mask=tgt_pad,
            tgt_mask=tgt_mask,
        )

        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_output.reshape(-1))

        non_pad = (tgt_output != PAD_ID).sum().item()
        total_loss += loss.item() * non_pad
        total_tokens += non_pad

    avg_loss = total_loss / max(total_tokens, 1)
    ppl = math.exp(min(avg_loss, 100))
    return avg_loss, ppl


print("Training and evaluation functions defined.")

## 9. Training Loop

Best checkpoint is saved whenever validation loss improves.

In [ ]:
CHECKPOINT_DIR = Path("../checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

history = {"train_loss": [], "val_loss": [], "train_ppl": [], "val_ppl": [], "lr": []}
best_val_loss = float("inf")

print(f"Training for {CONFIG['epochs']} epochs ...\n")
header = f"{'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>10} | {'Train PPL':>10} | {'Val PPL':>10} | {'LR':>12} | {'Time':>6}"
print(header)
print("-" * len(header))

for epoch in range(1, CONFIG["epochs"] + 1):
    t0 = time.time()

    train_loss, train_ppl = train_one_epoch(
        model, train_loader, optimizer, scheduler, criterion, DEVICE
    )
    val_loss, val_ppl = evaluate(model, val_loader, criterion, DEVICE)

    lr = optimizer.param_groups[0]["lr"]
    elapsed = time.time() - t0

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_ppl"].append(train_ppl)
    history["val_ppl"].append(val_ppl)
    history["lr"].append(lr)

    print(
        f"{epoch:5d} | {train_loss:10.4f} | {val_loss:10.4f} | "
        f"{train_ppl:10.2f} | {val_ppl:10.2f} | {lr:12.2e} | {elapsed:5.1f}s"
    )

    if USE_WANDB:
        wandb.log({
            "epoch": epoch,
            "train/loss": train_loss,
            "train/perplexity": train_ppl,
            "val/loss": val_loss,
            "val/perplexity": val_ppl,
            "lr": lr,
        })

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
                "val_ppl": val_ppl,
                "config": CONFIG,
            },
            CHECKPOINT_DIR / "best_baseline.pt",
        )
        print(f"        \u21b3 Best model saved (val_loss={val_loss:.4f})")

print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")

## 10. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs_range = range(1, len(history["train_loss"]) + 1)

axes[0].plot(epochs_range, history["train_loss"], label="Train", marker="o", markersize=4)
axes[0].plot(epochs_range, history["val_loss"],   label="Val",   marker="s", markersize=4)
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history["train_ppl"], label="Train", marker="o", markersize=4)
axes[1].plot(epochs_range, history["val_ppl"],   label="Val",   marker="s", markersize=4)
axes[1].set_title("Perplexity")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Perplexity")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_range, history["lr"], marker="o", markersize=4, color="green")
axes[2].set_title("Learning Rate Schedule")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("LR")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(CHECKPOINT_DIR / "training_curves_baseline.png", dpi=150, bbox_inches="tight")
plt.show()

if USE_WANDB:
    wandb.log({"training_curves": wandb.Image(str(CHECKPOINT_DIR / "training_curves_baseline.png"))})

print("Training curves saved.")

## 11. Generation Utilities

Three decoding strategies (proposal \u00a79 \u2014 Sweep 4 will explore these further):

| Strategy | Description |
|---|---|
| **Greedy** | `argmax` at every step |
| **Top-k** | sample from the top-k most likely tokens |
| **Nucleus (top-p)** | sample from the smallest set whose cumulative prob \u2265 p |

In [ ]:
@torch.no_grad()
def generate_story(
    model, tokenizer, encoder_input_text, max_len=200,
    strategy="greedy", temperature=1.0, top_k=50, top_p=0.9, device=DEVICE,
):
    """Autoregressive story generation given a formatted encoder input."""
    model.eval()

    enc_ids = [BOS_ID] + tokenizer.encode(encoder_input_text, add_special_tokens=False) + [EOS_ID]
    enc_tensor = torch.tensor([enc_ids], dtype=torch.long, device=device)
    enc_pad_mask = torch.zeros(1, len(enc_ids), dtype=torch.bool, device=device)

    memory = model.encode(enc_tensor, src_key_padding_mask=enc_pad_mask)

    dec_ids = [BOS_ID]

    for _ in range(max_len):
        tgt_tensor = torch.tensor([dec_ids], dtype=torch.long, device=device)
        tgt_mask = create_causal_mask(len(dec_ids), device)
        tgt_pad = torch.zeros(1, len(dec_ids), dtype=torch.bool, device=device)

        decoder_out = model.decode(
            tgt_tensor, memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_pad,
            memory_key_padding_mask=enc_pad_mask,
        )
        logits = model.output_proj(decoder_out[:, -1, :])

        if strategy == "greedy":
            next_token = logits.argmax(dim=-1).item()

        elif strategy == "top_k":
            logits = logits / temperature
            topk_logits, topk_idx = torch.topk(logits, top_k, dim=-1)
            probs = torch.softmax(topk_logits, dim=-1)
            next_token = topk_idx[0, torch.multinomial(probs, 1).item()].item()

        elif strategy == "top_p":
            logits = logits / temperature
            sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
            cum_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
            remove = cum_probs > top_p
            remove[:, 1:] = remove[:, :-1].clone()
            remove[:, 0] = False
            sorted_logits[remove] = float("-inf")
            probs = torch.softmax(sorted_logits, dim=-1)
            next_token = sorted_idx[0, torch.multinomial(probs, 1).item()].item()

        else:
            raise ValueError(f"Unknown strategy: {strategy}")

        if next_token == EOS_ID:
            break
        dec_ids.append(next_token)

    return tokenizer.decode(dec_ids[1:])


print("Generation utilities defined.")

## 12. Sample Generations

Load the best checkpoint and generate stories for a fixed set of (genre, title) prompts across all three decoding strategies.

In [ ]:
ckpt = torch.load(CHECKPOINT_DIR / "best_baseline.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Loaded best checkpoint (epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f})\n")

TEST_PROMPTS = [
    "[GENRE: Science Fiction] The Last Signal",
    "[GENRE: Fantasy] The Enchanted Forest",
    "[GENRE: Mystery] The Missing Witness",
    "[GENRE: Romance] Letters from Paris",
    "[GENRE: Horror] The Basement Door",
]

STRATEGIES = [
    ("greedy",  "Greedy",          {}),
    ("top_k",   "Top-k (k=50)",    {"top_k": 50, "temperature": 0.8}),
    ("top_p",   "Nucleus (p=0.9)", {"top_p": 0.9, "temperature": 0.8}),
]

wandb_rows = []

for prompt in TEST_PROMPTS:
    print("=" * 90)
    print(f"PROMPT: {prompt}")
    print("=" * 90)

    for key, label, kwargs in STRATEGIES:
        story = generate_story(
            model, tokenizer, prompt, max_len=150, strategy=key, **kwargs
        )
        preview = story[:300] + ("..." if len(story) > 300 else "")
        print(f"\n  [{label}]")
        print(f"  {preview}")
        wandb_rows.append([prompt, label, story[:500]])

    print()

if USE_WANDB:
    table = wandb.Table(
        columns=["Prompt", "Strategy", "Generated Story"], data=wandb_rows
    )
    wandb.log({"generated_samples": table})
    print("Sample generations logged to W&B.")

## 13. Save Final Checkpoint & W&B Artifact

The best model is versioned as a W&B Artifact (`baseline-v1`) for reproducible comparisons in future sweep notebooks (proposal \u00a76.3).

In [ ]:
torch.save(
    {"epoch": CONFIG["epochs"], "model_state_dict": model.state_dict(), "config": CONFIG},
    CHECKPOINT_DIR / "baseline_final.pt",
)
print(f"Final checkpoint saved to {CHECKPOINT_DIR / 'baseline_final.pt'}")

if USE_WANDB:
    artifact = wandb.Artifact(
        name="baseline-v1",
        type="model",
        description="Baseline encoder-decoder transformer for genre-conditioned story generation",
        metadata=CONFIG,
    )
    artifact.add_file(str(CHECKPOINT_DIR / "best_baseline.pt"))
    wandb.log_artifact(artifact)
    print("Model artifact logged to W&B.")

print(f"All checkpoints in: {CHECKPOINT_DIR.resolve()}")

## 14. Finish W&B Run

In [ ]:
if USE_WANDB:
    wandb.finish()
    print("W&B run finished.")
else:
    print("Local run complete (no W&B).")

## 15. Summary

| Item | Detail |
|---|---|
| **Architecture** | Vanilla encoder-decoder Transformer (PyTorch `nn.Transformer`) |
| **Parameters** | ~30 M (d=256, heads=8, layers=3+3, ff=512) |
| **Genre conditioning** | `[GENRE: <label>]` prefix in encoder input |
| **Training** | Teacher forcing, label-smoothed cross-entropy, AdamW, warmup |
| **Tracking** | W&B: loss, perplexity, LR, sample generations, model artifact |
| **Decoding** | Greedy, Top-k, Nucleus (top-p) |
| **Checkpoints** | `checkpoints/best_baseline.pt`, `checkpoints/baseline_final.pt` |

### Metrics logged to W&B
- `train/loss`, `train/perplexity`
- `val/loss`, `val/perplexity`
- `lr`
- `generated_samples` table
- `baseline-v1` model artifact

**Next notebook:** `4. Evaluation` \u2014 compute BLEU, ROUGE-L, Distinct-n, BERTScore on the test set using the saved baseline checkpoint.